In [ ]:
import os
import pandas as pd
from collections import defaultdict

In [ ]:
id_file="/project3/data/results/other-evals/ancientness/ancient_orfs_min100_only_headers.tsv"
tsv_dir="/project3/data/results/assembly-prokka-mmseqs-nr"

In [ ]:
# Load and filter headers efficiently
headers = pd.read_csv(id_file, header=None)[0]
headerset = {h for h in headers if "KapK" in h}

# Group headers by prefix
d2 = {}
for h in headerset:
    prefix = h.rsplit("_", 3)[0]
    d2.setdefault(prefix, []).append(h)  # cleaner than explicit key check

# Read TSVs and store query sets
d1 = {}
for filename in os.listdir(tsv_dir):
    if filename.endswith(".tsv") and "KapK" in filename:
        sample_id = filename.split(".raw-derep")[0]
        filepath = os.path.join(tsv_dir, filename)

        if not sample_id in d1.keys(): 
            try:
                df = pd.read_csv(filepath, sep="\t", header=None, usecols=[0])
                d1[sample_id] = set(df[0])
    
            except pd.errors.EmptyDataError:
                d1[sample_id] = set()

        else: 
            try:
                df = pd.read_csv(filepath, sep="\t", header=None, usecols=[0])
                d1[sample_id].update(set(df[0]))
    
            except pd.errors.EmptyDataError:
                continue
            
            



# Final intersection: only keep headers found in d1
final = {
    h
    for prefix, headers in d2.items()
    if prefix in d1
    for h in headers
    if h in d1[prefix]
}


In [ ]:
notinside = headerset - final

In [ ]:
len(final)

In [ ]:
len(notinside)